In [1]:
# ============================================================
# Row-level MIL + Set Transformer comparison
# ============================================================

import math
import pickle
import tqdm
from itertools import cycle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score



In [2]:

# ============================================================
# 0) Dataset / Collate
#    - global padding for row-raw ST models
# ============================================================

class TensorLabelListDatasetGlobalPad(Dataset):
    def __init__(self, data_list, global_H=None, global_W=None):
        self.data = data_list
        self.global_H = global_H if global_H is not None else max(x.shape[0] for x, _ in data_list)
        self.global_W = global_W if global_W is not None else max(x.shape[1] for x, _ in data_list)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]

        if not torch.is_tensor(x):
            x = torch.tensor(x)
        if not torch.is_tensor(y):
            y = torch.tensor(y)

        x = x.float()
        y = y.float().squeeze()

        H, W, C = x.shape
        assert C == 3, f"x must be (H,W,3), got {tuple(x.shape)}"
        assert y.dim() == 0, f"y must be scalar, got {tuple(y.shape)}"

        Xpad = torch.zeros(self.global_H, self.global_W, 3, dtype=x.dtype)
        Mpad = torch.zeros(self.global_H, self.global_W, dtype=torch.bool)

        Xpad[:H, :W] = x
        Mpad[:H, :W] = True

        return Xpad, Mpad, y


def collate_global_padded(batch):
    xs, masks, ys = zip(*batch)
    X = torch.stack(xs, dim=0)         # [B,Hg,Wg,3]
    mask = torch.stack(masks, dim=0)   # [B,Hg,Wg]
    y = torch.stack([yy.float() for yy in ys], dim=0)
    return X, mask, y


def infer_global_hw(*data_lists):
    global_H = 0
    global_W = 0
    for data_list in data_lists:
        if len(data_list) == 0:
            continue
        global_H = max(global_H, max(x.shape[0] for x, _ in data_list))
        global_W = max(global_W, max(x.shape[1] for x, _ in data_list))
    return global_H, global_W


def make_loaders_globalpad(train_list, val_list, batch_size=16, num_workers=0, pin_memory=False):
    global_H, global_W = infer_global_hw(train_list, val_list)

    train_ds = TensorLabelListDatasetGlobalPad(train_list, global_H=global_H, global_W=global_W)
    val_ds   = TensorLabelListDatasetGlobalPad(val_list,   global_H=global_H, global_W=global_W)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_global_padded,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_global_padded,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    return train_loader, val_loader, global_H, global_W


def make_single_loader_globalpad(data_list, global_H, global_W, batch_size=16, shuffle=True, num_workers=0, pin_memory=False):
    ds = TensorLabelListDatasetGlobalPad(data_list, global_H=global_H, global_W=global_W)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_global_padded,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )


def filter_normal_only(data_list):
    out = []
    for x, y in data_list:
        y_val = y.item() if torch.is_tensor(y) else float(y)
        if y_val == 0:
            out.append((x, y))
    return out


# ============================================================
# 1) Common utils
# ============================================================

def masked_mean(x: torch.Tensor, mask: torch.Tensor | None, dim: int, eps=1e-12):
    if mask is None:
        return x.mean(dim=dim)
    m = mask.float()
    while m.dim() < x.dim():
        m = m.unsqueeze(-1)
    num = (x * m).sum(dim=dim)
    den = m.sum(dim=dim).clamp_min(eps)
    return num / den


def masked_sum(x: torch.Tensor, mask: torch.Tensor | None, dim: int):
    if mask is None:
        return x.sum(dim=dim)
    m = mask.float()
    while m.dim() < x.dim():
        m = m.unsqueeze(-1)
    return (x * m).sum(dim=dim)


def masked_max(x: torch.Tensor, mask: torch.Tensor | None, dim: int):
    if mask is None:
        return x.max(dim=dim).values
    m = mask
    while m.dim() < x.dim():
        m = m.unsqueeze(-1)
    x = x.masked_fill(~m, float("-inf"))
    out = x.max(dim=dim).values
    out = torch.where(torch.isfinite(out), out, torch.zeros_like(out))
    return out


def masked_softmax(logits: torch.Tensor, mask: torch.Tensor | None, dim: int, eps=1e-12):
    if mask is None:
        return torch.softmax(logits, dim=dim)

    while mask.dim() < logits.dim():
        mask = mask.unsqueeze(-1)

    logits = logits.masked_fill(~mask, float("-inf"))
    attn = torch.softmax(logits, dim=dim)
    attn = torch.where(torch.isfinite(attn), attn, torch.zeros_like(attn))
    den = attn.sum(dim=dim, keepdim=True).clamp_min(eps)
    return attn / den


def masked_mse(a: torch.Tensor, b: torch.Tensor, mask: torch.Tensor | None):
    diff2 = (a - b) ** 2
    if mask is None:
        return diff2.mean()

    m = mask.float()
    while m.dim() < diff2.dim():
        m = m.unsqueeze(-1)

    num = (diff2 * m).sum()
    den = m.sum().clamp_min(1.0)
    return num / den


def flatten_valid(x_bhwc, mask_bhw=None):
    B, H, W, C = x_bhwc.shape
    x_bnc = x_bhwc.reshape(B, H * W, C)
    if mask_bhw is None:
        mask_bn = torch.ones(B, H * W, dtype=torch.bool, device=x_bhwc.device)
    else:
        mask_bn = mask_bhw.reshape(B, H * W).bool()
    return x_bnc, mask_bn


def grid_to_rows_for_mil(x_bhw3: torch.Tensor, mask_bhw: torch.Tensor | None = None, use_coords=True):
    """
    For MIL baselines:
      row -> instance
      row encoder handles [W,C] -> [D]

    returns:
      row_x   : [B,H,W,Csel]
      row_mask: [B,H]
      cellmask: [B,H,W]
    """
    if use_coords:
        row_x = x_bhw3
    else:
        row_x = x_bhw3[..., 0:1]

    if mask_bhw is None:
        B, H, W, _ = row_x.shape
        row_mask = torch.ones(B, H, dtype=torch.bool, device=row_x.device)
        cellmask = torch.ones(B, H, W, dtype=torch.bool, device=row_x.device)
    else:
        cellmask = mask_bhw.bool()
        row_mask = cellmask.any(dim=2)

    return row_x, row_mask, cellmask

def grid_to_score_rows(x_bhw3: torch.Tensor, mask_bhw: torch.Tensor | None = None):
    """
    AE input: always score only
    returns:
      row_score : [B,H,W]
      row_mask  : [B,H]
    """
    B, H, W, _ = x_bhw3.shape
    row_score = x_bhw3[..., 0:1].reshape(B, H, W)

    if mask_bhw is None:
        row_mask = torch.ones(B, H, dtype=torch.bool, device=x_bhw3.device)
    else:
        row_mask = mask_bhw.any(dim=2)

    return row_score, row_mask

def grid_to_coord_rows(x_bhw3: torch.Tensor, mask_bhw: torch.Tensor | None = None):
    """
    classifier optional extra input: row coords only
    returns:
      row_coords : [B,H,2W]
      row_mask   : [B,H]
    """
    B, H, W, _ = x_bhw3.shape
    row_coords = x_bhw3[..., 1:3].reshape(B, H, W * 2)

    if mask_bhw is None:
        row_mask = torch.ones(B, H, dtype=torch.bool, device=x_bhw3.device)
    else:
        row_mask = mask_bhw.any(dim=2)

    return row_coords, row_mask

def row_centers_from_coords(x_bhw3: torch.Tensor, mask_bhw: torch.Tensor | None = None):
    coords = x_bhw3[..., 1:3]
    if mask_bhw is None:
        return coords.mean(dim=2)

    m = mask_bhw.unsqueeze(-1).float()
    return (coords * m).sum(dim=2) / m.sum(dim=2).clamp_min(1.0)


# ============================================================
# 2) Metrics
# ============================================================

@torch.no_grad()
def eval_binary_metrics_from_logits(logits: torch.Tensor, y: torch.Tensor, thr=0.5):
    prob = torch.sigmoid(logits).detach().cpu().numpy()
    y_np = y.detach().cpu().numpy().astype(int)
    pred = (prob >= thr).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_np, pred, average="binary", zero_division=0
    )

    tp = int(((pred == 1) & (y_np == 1)).sum())
    tn = int(((pred == 0) & (y_np == 0)).sum())
    fp = int(((pred == 1) & (y_np == 0)).sum())
    fn = int(((pred == 0) & (y_np == 1)).sum())

    return {
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


@torch.no_grad()
def find_best_threshold(model, loader, device, thresholds=None):
    model.eval()
    all_logits, all_y = [], []

    for X, mask, y in loader:
        X = X.to(device)
        mask = mask.to(device)
        y = y.to(device)

        logits = model(X, mask)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())

    logits = torch.cat(all_logits, dim=0)
    y = torch.cat(all_y, dim=0)

    prob = torch.sigmoid(logits).numpy()
    y_np = y.numpy().astype(int)

    try:
        auroc = roc_auc_score(y_np, prob)
    except ValueError:
        auroc = float("nan")

    try:
        auprc = average_precision_score(y_np, prob)
    except ValueError:
        auprc = float("nan")

    if thresholds is None:
        thresholds = torch.linspace(0.05, 0.95, 19)

    best = {
        "thr": 0.5,
        "f1": -1.0,
        "precision": 0.0,
        "recall": 0.0,
        "AUROC": float(auroc),
        "AUPRC": float(auprc),
        "tp": 0, "tn": 0, "fp": 0, "fn": 0,
    }

    for thr in thresholds:
        m = eval_binary_metrics_from_logits(logits, y, thr=float(thr))
        if m["f1"] > best["f1"]:
            best = {
                "thr": float(thr),
                "f1": m["f1"],
                "precision": m["precision"],
                "recall": m["recall"],
                "tp": m["tp"],
                "tn": m["tn"],
                "fp": m["fp"],
                "fn": m["fn"],
                "AUROC": float(auroc),
                "AUPRC": float(auprc),
            }
    return best


# ============================================================
# 3) Row encoder for MIL baselines
#    meanmax pooling fixed as requested
# ============================================================

class RowCellEncoder(nn.Module):
    def __init__(self, in_dim=1, hidden=64, out_dim=64, dropout=0.1):
        super().__init__()
        self.cell_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
            nn.ReLU(inplace=True),
        )
        self.out_proj = nn.Linear(out_dim * 2, out_dim)

    def forward(self, row_x_bhwc, cellmask_bhw=None):
        h = self.cell_mlp(row_x_bhwc)  # [B,H,W,D]
        m1 = masked_mean(h, cellmask_bhw, dim=2)
        m2 = masked_max(h, cellmask_bhw, dim=2)
        row_emb = self.out_proj(torch.cat([m1, m2], dim=-1))
        return row_emb


# ============================================================
# 4) Set Transformer blocks
# ============================================================

class MAB(nn.Module):
    def __init__(self, dim_Q, dim_K, dim_V, h=4):
        super().__init__()
        assert dim_V % h == 0
        self.h = h
        self.fc_q = nn.Linear(dim_Q, dim_V)
        self.fc_k = nn.Linear(dim_K, dim_V)
        self.fc_v = nn.Linear(dim_K, dim_V)
        self.ln0 = nn.LayerNorm(dim_V)
        self.ln1 = nn.LayerNorm(dim_V)
        self.rff = nn.Sequential(
            nn.Linear(dim_V, dim_V * 2),
            nn.ReLU(inplace=True),
            nn.Linear(dim_V * 2, dim_V),
        )

    def forward(self, Q, K, mask_K=None):
        Q_ = self.fc_q(Q)
        K_ = self.fc_k(K)
        V_ = self.fc_v(K)

        B, Nq, DV = Q_.shape
        H = self.h
        d = DV // H

        def split(x):
            return x.view(B, -1, H, d).transpose(1, 2)

        Qh, Kh, Vh = split(Q_), split(K_), split(V_)
        att = (Qh @ Kh.transpose(-2, -1)) / math.sqrt(d)

        if mask_K is not None:
            att = att.masked_fill(~mask_K.unsqueeze(1).unsqueeze(2), float("-inf"))

        A = att.softmax(dim=-1)
        A = torch.where(torch.isfinite(A), A, torch.zeros_like(A))

        out = (A @ Vh).transpose(1, 2).contiguous().view(B, Nq, DV)
        out = self.ln0(out + Q_)
        out = self.ln1(self.rff(out) + out)
        return out


class PMA(nn.Module):
    def __init__(self, dim, k=1, h=4):
        super().__init__()
        self.S = nn.Parameter(torch.randn(k, dim))
        self.mab = MAB(dim, dim, dim, h)

    def forward(self, X, mask=None):
        B = X.size(0)
        S = self.S.unsqueeze(0).expand(B, -1, -1)
        return self.mab(S, X, mask_K=mask)


class ISAB(nn.Module):
    def __init__(self, dim_in, dim_out, m=16, h=4):
        super().__init__()
        self.I = nn.Parameter(torch.randn(m, dim_out))
        self.mab0 = MAB(dim_out, dim_in, dim_out, h)
        self.mab1 = MAB(dim_in, dim_out, dim_out, h)

    def forward(self, X, mask=None):
        B = X.size(0)
        I = self.I.unsqueeze(0).expand(B, -1, -1)
        H = self.mab0(I, X, mask_K=mask)
        return self.mab1(X, H, mask_K=None)


class ISAB_w_PMA(nn.Module):
    """
    Inducing points are replaced with PMA(X) + Linear
    """
    def __init__(self, dim_in, dim_out, m=16, h=4):
        super().__init__()
        self.pma_induce = PMA(dim_in, k=4, h=h)
        self.projI = nn.Linear(dim_in, dim_out)
        self.mab0 = MAB(dim_out, dim_in, dim_out, h)
        self.mab1 = MAB(dim_in, dim_out, dim_out, h)

    def forward(self, X, mask=None):
        I = self.projI(self.pma_induce(X, mask=mask))
        H = self.mab0(I, X, mask_K=mask)
        return self.mab1(X, H, mask_K=None)


class SetEncoder(nn.Module):
    def __init__(self, d_in=64, d_hid=128, n_isab=2, m_induce=16, heads=4, use_pma_isab=False):
        super().__init__()
        blk = ISAB_w_PMA if use_pma_isab else ISAB

        self.proj = nn.Linear(d_in, d_hid)
        self.blocks = nn.ModuleList([
            blk(d_hid, d_hid, m=m_induce, h=heads)
            for _ in range(n_isab)
        ])
        self.pma = PMA(d_hid, k=1, h=heads)

    def forward(self, X, mask=None):
        X = self.proj(X)
        for b in self.blocks:
            X = b(X, mask=mask)
        z_hidden = self.pma(X, mask=mask)   # [B,1,d_hid]
        return X, z_hidden


class TokenDecoder(nn.Module):
    def __init__(self, d_hid=128, d_out=64):
        super().__init__()
        self.dec = nn.Sequential(
            nn.Linear(d_hid, d_hid),
            nn.ReLU(inplace=True),
            nn.Linear(d_hid, d_out),
        )

    def forward(self, Z):
        return self.dec(Z)


class BinaryHead(nn.Module):
    def __init__(self, d_hid=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_hid, d_hid),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_hid, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)


# ============================================================
# 5) ST models
#    - raw row input
#    - classifier gets residual only
# ============================================================
class RowSTResidualClassifier(nn.Module):
    """
    AE:
      always reconstructs score row only

    Classifier:
      either residual only
      or concat([residual, coords])
    """
    requires_normal_loader = True

    def __init__(
        self,
        global_W: int,
        use_coords_in_classifier: bool,
        use_pma_isab: bool,
        d_hid=64,
        n_isab=2,
        m_induce=16,
        heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.global_W = global_W
        self.use_coords_in_classifier = use_coords_in_classifier

        # AE always sees score row only: dim = W
        ae_in_dim = global_W

        # classifier input dim
        # residual only -> W
        # residual + coords -> W + 2W = 3W
        cls_in_dim = global_W if not use_coords_in_classifier else global_W * 3

        self.ae_encoder = SetEncoder(
            d_in=ae_in_dim,
            d_hid=d_hid,
            n_isab=n_isab,
            m_induce=m_induce,
            heads=heads,
            use_pma_isab=use_pma_isab,
        )
        self.decoder = TokenDecoder(d_hid=d_hid, d_out=ae_in_dim)

        self.cls_encoder = SetEncoder(
            d_in=cls_in_dim,
            d_hid=d_hid,
            n_isab=n_isab,
            m_induce=m_induce,
            heads=heads,
            use_pma_isab=use_pma_isab,
        )
        self.cls_head = BinaryHead(d_hid=d_hid, dropout=dropout)

    def reconstruct(self, x_bhw3, mask_bhw=None):
        row_score, row_mask = grid_to_score_rows(x_bhw3, mask_bhw)   # [B,H,W]
        Ze, z_hidden = self.ae_encoder(row_score, mask=row_mask)
        row_score_hat = self.decoder(Ze)                             # [B,H,W]

        return {
            "row_score": row_score,
            "row_mask": row_mask,
            "Ze": Ze,
            "z_hidden": z_hidden,
            "row_score_hat": row_score_hat,
        }

    def make_classifier_input(self, x_bhw3, mask_bhw=None):
        rec = self.reconstruct(x_bhw3, mask_bhw)
        residual = (rec["row_score"] - rec["row_score_hat"]).abs()   # [B,H,W]

        if self.use_coords_in_classifier:
            row_coords, row_mask2 = grid_to_coord_rows(x_bhw3, mask_bhw)  # [B,H,2W]
            X_cls = torch.cat([residual, row_coords], dim=-1)             # [B,H,3W]
            row_mask = row_mask2
        else:
            X_cls = residual
            row_mask = rec["row_mask"]

        return X_cls, row_mask, rec

    def forward(self, x_bhw3, mask_bhw=None):
        X_cls, row_mask, _ = self.make_classifier_input(x_bhw3, mask_bhw)
        _, z_hidden = self.cls_encoder(X_cls, mask=row_mask)
        logits = self.cls_head(z_hidden.squeeze(1))
        return logits

    def compute_rec_loss(self, x_bhw3, mask_bhw=None):
        rec = self.reconstruct(x_bhw3, mask_bhw)
        L_rec = masked_mse(rec["row_score_hat"], rec["row_score"], rec["row_mask"])
        return {
            "total": L_rec,
            "rec": L_rec.detach(),
        }

    def compute_cls_loss(self, x_bhw3, y, mask_bhw=None, cls_weight=1.0, pos_weight=3.0):
        logits = self.forward(x_bhw3, mask_bhw)
        y = y.view_as(logits).float()
        pos_w = torch.tensor([pos_weight], device=logits.device)
        L_cls = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_w)
        return {
            "total": cls_weight * L_cls,
            "logits": logits,
            "cls": L_cls.detach(),
        }

    def compute_losses(self, x_bhw3, y, mask_bhw=None, **kwargs):
        return self.compute_cls_loss(x_bhw3, y, mask_bhw=mask_bhw, **kwargs)


# ============================================================
# 6) Shared row MIL backbone for baselines
# ============================================================

class SharedRowMILBackbone(nn.Module):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, dropout=0.1):
        super().__init__()
        in_dim = 3 if use_coords else 1
        self.use_coords = use_coords
        self.row_encoder = RowCellEncoder(
            in_dim=in_dim,
            hidden=row_hidden,
            out_dim=row_dim,
            dropout=dropout,
        )

    def encode_rows(self, x_bhw3, mask_bhw=None):
        row_x, row_mask, cellmask = grid_to_rows_for_mil(x_bhw3, mask_bhw, use_coords=self.use_coords)
        row_emb = self.row_encoder(row_x, cellmask_bhw=cellmask)
        centers = row_centers_from_coords(x_bhw3, mask_bhw)
        return row_emb, row_mask, centers


# ============================================================
# 7) Baselines
# ============================================================

class DeepSetsRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.rho = nn.Sequential(
            nn.Linear(row_dim, row_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(row_dim, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)
        g = masked_mean(row_emb, row_mask, dim=1)
        return self.rho(g).squeeze(-1)


class ABMILRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, attn=32, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.attn_V = nn.Linear(row_dim, attn)
        self.attn_w = nn.Linear(attn, 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(row_dim, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)
        a = self.attn_w(torch.tanh(self.attn_V(row_emb))).squeeze(-1)
        alpha = masked_softmax(a, row_mask, dim=1)
        z = torch.sum(row_emb * alpha.unsqueeze(-1), dim=1)
        return self.head(z).squeeze(-1)


class GatedMILRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, attn=32, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.attn_V = nn.Linear(row_dim, attn)
        self.attn_U = nn.Linear(row_dim, attn)
        self.attn_w = nn.Linear(attn, 1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(row_dim, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)
        v = torch.tanh(self.attn_V(row_emb))
        u = torch.sigmoid(self.attn_U(row_emb))
        a = self.attn_w(v * u).squeeze(-1)
        alpha = masked_softmax(a, row_mask, dim=1)
        z = torch.sum(row_emb * alpha.unsqueeze(-1), dim=1)
        return self.head(z).squeeze(-1)


class DSMILRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, attn_dim=32, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.inst_cls = nn.Linear(row_dim, 1)
        self.q_proj = nn.Linear(row_dim, attn_dim)
        self.v_proj = nn.Linear(row_dim, row_dim)
        self.bag_cls = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(row_dim, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)
        B, H, D = row_emb.shape

        inst_logits = self.inst_cls(row_emb).squeeze(-1)
        masked_inst_logits = inst_logits.masked_fill(~row_mask, float("-inf"))
        crit_idx = masked_inst_logits.argmax(dim=1)

        batch_idx = torch.arange(B, device=row_emb.device)
        crit_h = row_emb[batch_idx, crit_idx]

        q_all = self.q_proj(row_emb)
        q_crit = self.q_proj(crit_h).unsqueeze(1)
        v_all = self.v_proj(row_emb)

        scores = (q_all * q_crit).sum(dim=-1) / math.sqrt(q_all.size(-1))
        attn = masked_softmax(scores, row_mask, dim=1)

        z = torch.sum(v_all * attn.unsqueeze(-1), dim=1)
        return self.bag_cls(z).squeeze(-1)


class DeepSVDDRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=64, rep=32, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.rep = nn.Sequential(
            nn.Linear(row_dim, row_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(row_dim, rep),
        )
        self.head = nn.Sequential(
            nn.Linear(rep, row_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(row_dim, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)
        z = self.rep(row_emb)
        g = masked_mean(z, row_mask, dim=1)
        return self.head(g).squeeze(-1)


# ============================================================
# 8) Row PointNet++-style baseline
# ============================================================

def pairwise_dist(a, b):
    aa = (a ** 2).sum(-1, keepdim=True)
    bb = (b ** 2).sum(-1, keepdim=True).transpose(1, 2)
    ab = a @ b.transpose(1, 2)
    return aa + bb - 2 * ab


def index_points(x, idx):
    B = x.size(0)
    if idx.dim() == 2:
        return x[torch.arange(B, device=x.device).unsqueeze(1), idx]
    return x[torch.arange(B, device=x.device).unsqueeze(1).unsqueeze(2), idx]


def farthest_point_sampling(coords, M, mask=None):
    B, N, _ = coords.shape
    device = coords.device
    valid = mask if mask is not None else torch.ones(B, N, dtype=torch.bool, device=device)
    idx = torch.zeros(B, M, dtype=torch.long, device=device)

    first = valid.float().argmax(dim=1)
    idx[:, 0] = first

    dist = torch.full((B, N), float("inf"), device=device)
    sel = coords[torch.arange(B, device=device), idx[:, 0]]
    d = ((coords - sel.unsqueeze(1)) ** 2).sum(-1)
    dist = torch.minimum(dist, d)
    dist = dist.masked_fill(~valid, float("-inf"))

    for i in range(1, M):
        far = dist.argmax(dim=1)
        idx[:, i] = far
        sel = coords[torch.arange(B, device=device), far]
        d = ((coords - sel.unsqueeze(1)) ** 2).sum(-1)
        dist = torch.minimum(dist, d)
        dist = dist.masked_fill(~valid, float("-inf"))
    return idx


def knn_group_pn(coords, centers, k, mask=None):
    dist = pairwise_dist(centers, coords)
    if mask is not None:
        dist = dist.masked_fill(~mask.unsqueeze(1), float("inf"))
    return dist.topk(k=min(k, coords.size(1)), largest=False, dim=-1).indices


class SharedMLP(nn.Module):
    def __init__(self, in_dim, dims, dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for d in dims:
            layers += [nn.Linear(prev, d), nn.ReLU(inplace=True)]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = d
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class SA2D(nn.Module):
    def __init__(self, M, k, in_feat, mlp_dims, dropout=0.1):
        super().__init__()
        self.M = M
        self.k = k
        self.local = SharedMLP(2 + in_feat, mlp_dims, dropout)

    def forward(self, coords, feat, mask=None):
        B, N, _ = coords.shape
        M = min(self.M, N)
        fps_idx = farthest_point_sampling(coords, M, mask=mask)
        cent = index_points(coords, fps_idx)
        knn_idx = knn_group_pn(coords, cent, self.k, mask=mask)
        neigh_c = index_points(coords, knn_idx)
        neigh_f = index_points(feat, knn_idx)
        rel = neigh_c - cent.unsqueeze(2)
        x = torch.cat([rel, neigh_f], dim=-1)
        h = self.local(x)
        new_f = h.max(dim=2).values
        new_mask = None if mask is None else mask.any(dim=1, keepdim=True).expand(B, M)
        return cent, new_f, new_mask


class PointNetPPRowBinary(SharedRowMILBackbone):
    def __init__(self, use_coords=True, row_hidden=64, row_dim=32, dropout=0.1):
        super().__init__(use_coords, row_hidden, row_dim, dropout)
        self.sa1 = SA2D(M=32, k=8, in_feat=row_dim, mlp_dims=[32, 32, 64], dropout=dropout)
        self.sa2 = SA2D(M=8, k=8, in_feat=64, mlp_dims=[64, 64, 96], dropout=dropout)
        self.sa3 = SA2D(M=1, k=8, in_feat=96, mlp_dims=[96, 128, 128], dropout=dropout)

        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, centers = self.encode_rows(x_bhw3, mask_bhw)
        coords, feat, mask = centers, row_emb, row_mask
        coords, feat, mask = self.sa1(coords, feat, mask)
        coords, feat, mask = self.sa2(coords, feat, mask)
        coords, feat, mask = self.sa3(coords, feat, mask)
        g = feat.squeeze(1)
        return self.head(g).squeeze(-1)


# ============================================================
# 9) Conv baselines
# ============================================================

class ConvTopKBinary(nn.Module):
    def __init__(
        self,
        hidden=32,
        depth=3,
        dropout=0.1,
        pooling="topk",       # "topk" or "lse"
        topk_frac=0.15,
        lse_tau=0.2,
        pos_weight=1.5,
        use_xy=False,
        aux_map_loss_weight=0.0,
    ):
        super().__init__()
        assert pooling in ("topk", "lse")
        self.pooling = pooling
        self.topk_frac = topk_frac
        self.lse_tau = lse_tau
        self.pos_weight = float(pos_weight)
        self.use_xy = use_xy
        self.aux_map_loss_weight = float(aux_map_loss_weight)

        c_in = 3 if use_xy else 1

        layers = []
        c = c_in
        for _ in range(depth):
            layers += [
                nn.Conv2d(c, hidden, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(hidden),
                nn.ReLU(inplace=True),
            ]
            if dropout > 0:
                layers.append(nn.Dropout2d(dropout))
            c = hidden
        self.backbone = nn.Sequential(*layers)

        self.score_head = nn.Conv2d(hidden, 1, kernel_size=1)
        self.global_head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self.fuse = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(inplace=True),
            nn.Linear(8, 1),
        )

    def _masked_topk_pool(self, score_map, mask, frac):
        B, H, W = score_map.shape
        flat_scores = score_map.view(B, H * W)
        flat_mask = mask.view(B, H * W)

        flat_scores = flat_scores.masked_fill(~flat_mask, -1e9)
        valid_counts = flat_mask.sum(dim=1)

        pooled = []
        for b in range(B):
            n_valid = int(valid_counts[b].item())
            k = max(1, int(round(n_valid * frac)))
            vals = flat_scores[b].topk(k, dim=0).values
            pooled.append(vals.mean())
        return torch.stack(pooled, dim=0)

    def _masked_lse_pool(self, score_map, mask, tau):
        B, H, W = score_map.shape
        flat_scores = score_map.view(B, H * W)
        flat_mask = mask.view(B, H * W)
        flat_scores = flat_scores.masked_fill(~flat_mask, -1e9)
        return tau * torch.logsumexp(flat_scores / tau, dim=1)

    def forward(self, x_bhw3, mask_bhw=None, return_aux=False):
        if mask_bhw is None:
            mask_bhw = torch.ones(x_bhw3.shape[:3], dtype=torch.bool, device=x_bhw3.device)

        x = x_bhw3.permute(0, 3, 1, 2) if self.use_xy else x_bhw3[..., 0:1].permute(0, 3, 1, 2)

        feat = self.backbone(x)
        score_map = self.score_head(feat).squeeze(1)

        if self.pooling == "topk":
            local_logit = self._masked_topk_pool(score_map, mask_bhw, self.topk_frac)
        else:
            local_logit = self._masked_lse_pool(score_map, mask_bhw, self.lse_tau)

        feat_hw = feat.permute(0, 2, 3, 1)
        m = mask_bhw.unsqueeze(-1).float()
        feat_mean = (feat_hw * m).sum(dim=(1, 2)) / m.sum(dim=(1, 2)).clamp_min(1.0)
        global_logit = self.global_head(feat_mean).squeeze(-1)

        fuse_in = torch.stack([local_logit, global_logit], dim=-1)
        final_logit = self.fuse(fuse_in).squeeze(-1)

        if return_aux:
            return final_logit, {
                "score_map": score_map,
                "local_logit": local_logit,
                "global_logit": global_logit,
            }
        return final_logit

    def compute_losses(self, x_bhw3, y, mask_bhw=None, **kwargs):
        logits, aux = self.forward(x_bhw3, mask_bhw, return_aux=True)
        y = y.view_as(logits).float()
        pos_w = torch.tensor([self.pos_weight], device=logits.device)
        L_cls = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_w)

        total = L_cls

        if self.aux_map_loss_weight > 0:
            local_loss = F.binary_cross_entropy_with_logits(aux["local_logit"], y, pos_weight=pos_w)
            global_loss = F.binary_cross_entropy_with_logits(aux["global_logit"], y, pos_weight=pos_w)
            total = total + self.aux_map_loss_weight * (local_loss + global_loss)

        return {"total": total, "logits": logits, "cls": L_cls.detach()}


class FF(nn.Module):
    def __init__(self, d, dff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, dff),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(dff, d),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class PICASOBlock(nn.Module):
    """
    Template <-> instance cross-attention block
    """
    def __init__(self, d, heads, dff, dropout=0.1):
        super().__init__()
        self.attn_T = nn.MultiheadAttention(
            embed_dim=d,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_X = nn.MultiheadAttention(
            embed_dim=d,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )

        self.lnT1 = nn.LayerNorm(d)
        self.lnT2 = nn.LayerNorm(d)
        self.lnX1 = nn.LayerNorm(d)
        self.lnX2 = nn.LayerNorm(d)

        self.ffT = FF(d, dff, dropout)
        self.ffX = FF(d, dff, dropout)

    def forward(self, X, T, mask_X=None):
        # mask_X: [B,H], True=valid
        key_padding_mask = None if mask_X is None else ~mask_X  # True=padding for MHA

        T2, _ = self.attn_T(
            query=T,
            key=X,
            value=X,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        T = self.lnT1(T + T2)
        T = self.lnT2(T + self.ffT(T))

        X2, _ = self.attn_X(
            query=X,
            key=T,
            value=T,
            need_weights=False,
        )
        X = self.lnX1(X + X2)
        X = self.lnX2(X + self.ffX(X))

        return X, T


class PICASORowBinary(SharedRowMILBackbone):
    """
    Row-level MIL baseline with PICASO-style dynamic templates
    """
    def __init__(
        self,
        use_coords=True,
        row_hidden=64,
        row_dim=64,
        d=64,
        heads=4,
        dff=128,
        layers=2,
        templates=8,
        pooling="mean",
        dropout=0.1,
    ):
        super().__init__(use_coords=use_coords, row_hidden=row_hidden, row_dim=row_dim, dropout=dropout)
        assert pooling in ("mean", "sum")
        assert d == row_dim, "For simplicity, use d == row_dim"

        self.pooling = pooling
        self.T0 = nn.Parameter(torch.randn(templates, d) / math.sqrt(d))
        self.blocks = nn.ModuleList([
            PICASOBlock(d=d, heads=heads, dff=dff, dropout=dropout)
            for _ in range(layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, 1),
        )

    def forward(self, x_bhw3, mask_bhw=None):
        row_emb, row_mask, _ = self.encode_rows(x_bhw3, mask_bhw)   # [B,H,D], [B,H]
        B = row_emb.size(0)

        X = row_emb
        T = self.T0.unsqueeze(0).expand(B, -1, -1).contiguous()     # [B,K,D]

        for blk in self.blocks:
            X, T = blk(X, T, mask_X=row_mask)

        if self.pooling == "mean":
            g = masked_mean(X, row_mask, dim=1)
        else:
            g = masked_sum(X, row_mask, dim=1)

        return self.head(g).squeeze(-1)
    
# ============================================================
# 10) Wrapper for supervised baselines
# ============================================================

class SupervisedADWrapper(nn.Module):
    def __init__(self, backbone: nn.Module, pos_weight=1.5):
        super().__init__()
        self.backbone = backbone
        self.pos_weight = float(pos_weight)

    def forward(self, x_bhw3, mask_bhw=None):
        return self.backbone(x_bhw3, mask_bhw)

    def compute_losses(self, x_bhw3, y, mask_bhw=None, **kwargs):
        logits = self.forward(x_bhw3, mask_bhw)
        y = y.view_as(logits).float()
        pos_w = torch.tensor([self.pos_weight], device=logits.device)
        L_cls = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_w)
        return {"total": L_cls, "logits": logits, "cls": L_cls.detach()}


# ============================================================
# 11) Build model
# ============================================================
def build_model(name: str, device: str, global_W: int):
    name = name.lower()

    if name == "st_row_residual":
        return RowSTResidualClassifier(
            global_W=global_W,
            use_coords_in_classifier=False,
            use_pma_isab=False,
            d_hid=32,
            n_isab=2,
            m_induce=16,
            heads=8,
            dropout=0.1,
        ).to(device)

    if name == "st_row_residual_w_pma":
        return RowSTResidualClassifier(
            global_W=global_W,
            use_coords_in_classifier=False,
            use_pma_isab=True,
            d_hid=32,
            n_isab=2,
            m_induce=16,
            heads=8,
            dropout=0.1,
        ).to(device)

    if name == "st_row_residual_coords":
        return RowSTResidualClassifier(
            global_W=global_W,
            use_coords_in_classifier=True,
            use_pma_isab=False,
            d_hid=32,
            n_isab=2,
            m_induce=16,
            heads=8,
            dropout=0.1,
        ).to(device)

    if name == "st_row_residual_coords_w_pma":
        return RowSTResidualClassifier(
            global_W=global_W,
            use_coords_in_classifier=True,
            use_pma_isab=True,
            d_hid=32,
            n_isab=2,
            m_induce=16,
            heads=8,
            dropout=0.1,
        ).to(device)

    
    # ---- Row MIL baselines ----
    if name == "deepsvdd":
        return SupervisedADWrapper(
            DeepSVDDRowBinary(use_coords=True, row_hidden=64, row_dim=64, rep=32, dropout=0.1)
        ).to(device)

    if name == "abmil":
        return SupervisedADWrapper(
            ABMILRowBinary(use_coords=True, row_hidden=64, row_dim=64, attn=32, dropout=0.1)
        ).to(device)

    if name == "gatedmil":
        return SupervisedADWrapper(
            GatedMILRowBinary(use_coords=True, row_hidden=64, row_dim=64, attn=32, dropout=0.1)
        ).to(device)

    if name == "dsmil":
        return SupervisedADWrapper(
            DSMILRowBinary(use_coords=True, row_hidden=64, row_dim=64, attn_dim=32, dropout=0.1)
        ).to(device)

    if name == "pointnetpp":
        return SupervisedADWrapper(
            PointNetPPRowBinary(use_coords=True, row_hidden=64, row_dim=32, dropout=0.1)
        ).to(device)

    if name == "deepsets":
        return SupervisedADWrapper(
            DeepSetsRowBinary(use_coords=True, row_hidden=64, row_dim=64, dropout=0.1)
        ).to(device)

    # ---- 2D conv baselines ----
    if name == "conv_lse":
        return ConvTopKBinary(
            hidden=32,
            depth=2,
            dropout=0.1,
            pooling="lse",
            topk_frac=0.15,
            lse_tau=0.2,
            pos_weight=3.0,
            use_xy=False,
            aux_map_loss_weight=0.0,
        ).to(device)

    if name == "conv_topk":
        return ConvTopKBinary(
            hidden=32,
            depth=2,
            dropout=0.1,
            pooling="topk",
            topk_frac=0.15,
            lse_tau=0.2,
            pos_weight=3.0,
            use_xy=False,
            aux_map_loss_weight=0.0,
        ).to(device)
    if name == "picaso":
        return SupervisedADWrapper(
            PICASORowBinary(
                use_coords=True,
                row_hidden=32,
                row_dim=32,
                d=32,
                heads=8,
                dff=64,
                layers=2,
                templates=8,
                pooling="mean",
                dropout=0.1,
            )
        ).to(device)
    raise ValueError(f"unknown model: {name}")

# ============================================================
# 12) Train helpers
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_clip=None,
    cls_weight=1.0,
):
    model.train()
    total_loss = 0.0
    n_total = 0

    for X, mask, y in loader:
        X = X.to(device)
        mask = mask.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        out = model.compute_losses(X, y, mask_bhw=mask, cls_weight=cls_weight)
        loss = out["total"]
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        n_total += bs

    return total_loss / max(n_total, 1)


def train_one_epoch_rec_only(
    model,
    normal_loader,
    optimizer,
    device,
    grad_clip=None,
):
    model.train()
    total_loss = 0.0
    n_total = 0

    for Xn, maskn, _ in normal_loader:
        Xn = Xn.to(device)
        maskn = maskn.to(device)

        optimizer.zero_grad(set_to_none=True)
        out = model.compute_rec_loss(Xn, mask_bhw=maskn)
        loss = out["total"]
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        bs = Xn.size(0)
        total_loss += loss.item() * bs
        n_total += bs

    return total_loss / max(n_total, 1)


def train_one_epoch_dual(
    model,
    normal_loader,
    mixed_loader,
    optimizer,
    device,
    rec_weight=1.0,
    cls_weight=1.0,
    grad_clip=None,
):
    model.train()
    total_loss = 0.0
    n_total = 0

    normal_iter = cycle(normal_loader)

    for Xm, maskm, ym in mixed_loader:
        Xn, maskn, _ = next(normal_iter)

        Xm = Xm.to(device)
        maskm = maskm.to(device)
        ym = ym.to(device)

        Xn = Xn.to(device)
        maskn = maskn.to(device)

        optimizer.zero_grad(set_to_none=True)

        out_rec = model.compute_rec_loss(Xn, mask_bhw=maskn)
        out_cls = model.compute_cls_loss(Xm, ym, mask_bhw=maskm, cls_weight=cls_weight)

        loss = rec_weight * out_rec["total"] + out_cls["total"]
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        bs = ym.size(0)
        total_loss += loss.item() * bs
        n_total += bs

    return total_loss / max(n_total, 1)


# ============================================================
# 13) Training runner
# ============================================================

def run_training(
    model,
    model_name,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=2e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    times=0,
    normal_train_loader=None,
    st_warmup_epochs=10,
    rec_weight=1.0,
    cls_weight=1.0,
    save_prefix=None,
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_f1 = -1.0
    best_thr = 0.5

    use_dual_loader = getattr(model, "requires_normal_loader", False)

    if use_dual_loader and normal_train_loader is None:
        raise ValueError(f"{model_name} needs normal_train_loader, but got None")

    history = {
        "epoch": [], "phase": [], "train": [], "thr": [],
        "prec": [], "rec": [], "f1": [], "auroc": [], "auprc": []
    }

    for ep in tqdm.tqdm(range(epochs)):
        if use_dual_loader:
            if ep < st_warmup_epochs:
                phase = "warmup_rec"
                tr_loss = train_one_epoch_rec_only(
                    model=model,
                    normal_loader=normal_train_loader,
                    optimizer=optimizer,
                    device=device,
                    grad_clip=grad_clip,
                )
            else:
                phase = "joint"
                tr_loss = train_one_epoch_dual(
                    model=model,
                    normal_loader=normal_train_loader,
                    mixed_loader=train_loader,
                    optimizer=optimizer,
                    device=device,
                    rec_weight=rec_weight,
                    cls_weight=cls_weight,
                    grad_clip=grad_clip,
                )
        else:
            phase = "sup"
            tr_loss = train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                device=device,
                grad_clip=grad_clip,
                cls_weight=cls_weight,
            )

        with torch.no_grad():
            m = find_best_threshold(model, val_loader, device)
            thr = m["thr"]
            va_f1 = m["f1"]
            va_p = m["precision"]
            va_r = m["recall"]
            va_auroc = m["AUROC"]
            va_auprc = m["AUPRC"]

        history["epoch"].append(f"{ep:03d}")
        history["phase"].append(phase)
        history["train"].append(f"{tr_loss:.4f}")
        history["thr"].append(f"{thr:.2f}")
        history["prec"].append(f"{va_p:.4f}")
        history["rec"].append(f"{va_r:.4f}")
        history["f1"].append(f"{va_f1:.4f}")
        history["auroc"].append(f"{va_auroc:.4f}")
        history["auprc"].append(f"{va_auprc:.4f}")

        if save_prefix is not None:
            with open(f"{save_prefix}/score/dictLoss_{model_name}_{times}.pkl", "wb") as f:
                pickle.dump(history, f)

        if va_f1 > best_f1:
            best_f1, best_thr = va_f1, thr
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

            if save_prefix is not None:
                torch.save(best_state, f"{save_prefix}/model/{model_name}_{times}_best.pt")

    return best_state, best_f1, best_thr


# ============================================================
# 14) compare_all
# ============================================================

def compare_all(
    model_names,
    train_list,
    val_list,
    device="cuda",
    batch_size=16,
    epochs=100,
    lr=2e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    times=0,
    st_warmup_epochs=10,
    rec_weight=1.0,
    cls_weight=1.0,
    num_workers=0,
    pin_memory=False,
    save_prefix=None,
):
    train_loader, val_loader, global_H, global_W = make_loaders_globalpad(
        train_list, val_list,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    normal_train_list = filter_normal_only(train_list)
    normal_train_loader = make_single_loader_globalpad(
        normal_train_list,
        global_H=global_H,
        global_W=global_W,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    results = {}

    for model_name in model_names:
        print(f"\n==== train: {model_name} ====")
        model = build_model(model_name, device, global_W=global_W)

        best_state, best_f1, best_thr = run_training(
            model=model,
            model_name=model_name,
            train_loader=train_loader,
            val_loader=val_loader,
            normal_train_loader=normal_train_loader,
            device=device,
            epochs=epochs,
            lr=lr,
            weight_decay=weight_decay,
            grad_clip=grad_clip,
            times=times,
            st_warmup_epochs=st_warmup_epochs,
            rec_weight=rec_weight,
            cls_weight=cls_weight,
            save_prefix=save_prefix,
        )

        results[model_name] = {
            "best_f1": best_f1,
            "best_thr": best_thr,
            "best_state": best_state,
        }

    return results


# ============================================================
# 15) Example model list
# ============================================================
DEFAULT_MODEL_NAMES = [
    "st_row_residual",
    "st_row_residual_w_pma",
    "st_row_residual_coords",
    "st_row_residual_coords_w_pma",
    "deepsvdd",
    "abmil",
    "gatedmil",
    "dsmil",
    "picaso",
    "pointnetpp",
    "deepsets",
    "conv_lse",
    "conv_topk",
]


# ============================================================
# 16) Example usage
# ============================================================


In [4]:
names = DEFAULT_MODEL_NAMES

for times in range(1, 11):
    with open(f"data/common/trainSet{times}.pkl", "rb") as f:
        train_list = pickle.load(f)
    with open(f"data/common/valSet{times}.pkl", "rb") as f:
        val_list = pickle.load(f)

    results = compare_all(
        model_names=names,
        train_list=train_list,
        val_list=val_list,
        device="cuda",
        batch_size=256,
        epochs=200,
        lr=2e-4,
        weight_decay=1e-4,
        grad_clip=1.0,
        times=times,
        st_warmup_epochs=10,
        rec_weight=1.0,
        cls_weight=1.0,
        num_workers=0,
        pin_memory=False,
        save_prefix="data/260402_ST_MIL",
    )

    # for k, v in results.items():
    #     print(k, v["best_f1"], v["best_thr"])


==== train: st_row_residual ====


100%|██████████| 200/200 [01:56<00:00,  1.72it/s]



==== train: st_row_residual_w_pma ====


100%|██████████| 200/200 [02:26<00:00,  1.36it/s]



==== train: st_row_residual_coords ====


...

100%|██████████| 200/200 [02:48<00:00,  1.18it/s]



==== train: conv_lse ====


100%|██████████| 200/200 [06:02<00:00,  1.81s/it]



==== train: conv_topk ====


100%|██████████| 200/200 [07:59<00:00,  2.40s/it]
